# PyTorch, Part 1: Classification with No Hidden Layer

CSCI 6353 · Topic 24. First contact with PyTorch: the binary and multiclass classifiers of Topics 21-23, reproduced in a few lines. Input straight to output — no hidden layer, no neural network yet.

## Binary classification (the Topic 21/22 problem)

One feature, labels 0/1. `nn.Linear(1,1)` is exactly w·x + b; `BCEWithLogitsLoss` bundles the sigmoid + log-loss; `SGD` is our gradient descent.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

X = torch.tensor([[2.], [3.], [4.], [5.], [6.], [-2.], [-3.], [-4.], [-5.], [-6.]])
y = torch.tensor([[1.], [1.], [1.], [1.], [1.], [0.], [0.], [0.], [0.], [0.]])

torch.manual_seed(0)
model = nn.Linear(1, 1)                        # w*x + b : no hidden layer
criterion = nn.BCEWithLogitsLoss()             # sigmoid + log-loss, bundled
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(2000):
    optimizer.zero_grad()                      # clear old gradients
    loss = criterion(model(X), y)              # forward + measure error
    loss.backward()                            # autograd computes gradients
    optimizer.step()                           # gradient-descent update

probs = torch.sigmoid(model(X)).detach().flatten()
print("probabilities:", probs.round(decimals=3).tolist())
print("accuracy:", ((probs >= 0.5).float() == y.flatten()).float().mean().item())

## Multiclass (the Topic 23 problem), still no hidden layer

Only three things change: `nn.Linear(4, 3)` (one score per class), `CrossEntropyLoss` (softmax + cross-entropy; labels are plain class indices, NOT one-hot), and `argmax` to predict.

In [ ]:
trainX = torch.tensor([[0.1,0.2,0.3,0.2],[0.5,0.4,0.3,0.7],[0.3,0.7,0.4,0.1],
                       [0.2,0.8,0.9,0.3],[1.1,0.5,0.2,0.9],
                       [4.3,5.3,4.7,4.2],[4.5,5.1,5.3,4.4],[5.1,4.8,5.1,4.6],
                       [4.9,4.6,4.9,4.3],[5.4,5.5,4.3,4.7],
                       [10.1,10.2,10.3,11.3],[11.3,11.2,11.1,10.3],[12.5,12.3,12.1,11.4],
                       [11.7,11.8,11.2,12.8],[13.1,10.2,12.4,11.7]])
trainY = torch.tensor([0]*5 + [1]*5 + [2]*5)   # class indices — no one-hot!

testX = torch.tensor([[0.5,0.4,0.6,0.5],[5.4,5.6,5.5,5.2],[11.7,11.6,11.5,11.4]])
testY = torch.tensor([0, 1, 2])

torch.manual_seed(0)
model = nn.Linear(4, 3)                        # 4 features -> 3 class scores
criterion = nn.CrossEntropyLoss()              # softmax + cross-entropy, bundled
optimizer = optim.SGD(model.parameters(), lr=0.05)

for epoch in range(2000):                      # the loop is IDENTICAL
    optimizer.zero_grad()
    loss = criterion(model(trainX), trainY)
    loss.backward()
    optimizer.step()

pred = model(testX).argmax(1)                  # largest score wins
print("test predictions:", pred.tolist(), " true:", testY.tolist())
print("accuracy:", (pred == testY).float().mean().item())

## Peek inside

The model really is just a weight matrix and a bias — the same W (4x3) and b (3) we trained by hand in Topic 23.

In [ ]:
print("W:", model.weight.detach().round(decimals=3))
print("b:", model.bias.detach().round(decimals=3))
print("softmax of test scores (rows sum to 1):")
print(torch.softmax(model(testX), dim=1).detach().round(decimals=3))